---
title: Red Neuronal desde Primeros Principios con PyTorch
subject: Aprendizaje Profundo
subtitle: Explicación paso a paso
short_title: RN con PyTorch
authors:
  - name: Jorge Anais
    orcid: 0000-0001-9051-1338
    email: jrganais@gmail.com
license: MIT
---

**Objetivo**: Programar una red neuronal usando los principios teóricos para resolver una tarea de clasificación.

**Referencias**:
  - [PyTorch Doc](https://https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
  - [Jorge Perez YT](https://youtu.be/y6aD4WG-rOw?si=fyNnilzbXLdWPFyr)

## Imporación librerias

In [7]:
import sys
import time

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader


# Fijamos una semilla para que los experimentos sean reproducibles
t_cg = torch.manual_seed(1234)

## Contexto del caso

El problema que abordaremos en este ejercicio, es la clasificación de los dígitos "0" y "1" a partir de una imágenes que contine ejemplos escritos a mano.

![Ejemplos](https://github.com/jorgeanais/mlt2202/blob/main/ea1/s3/assets/ejemplos.png?raw=true)


Para ello utilizaremos parte del conjunto de datos MNIST que se provee en un archivo `CSV`. La primera columna corresponde a la etiqueta (0 ó 1), mientras que las siguientes columnas representan el valor de cada pixel de la imágen. Cada pixel puede tener un valor entre 0 y 255. La figura siguiente muestra una representación de como está guardada la información de cada dígito.

![RepresentacionNumerica](https://github.com/jorgeanais/mlt2202/blob/main/ea1/s3/assets/representacion_numero.png?raw=true)


## Funciones de activación y entropía cruzada binaria

Formulario funciones de activación y su derivada  

- *Sigmoide*:

$$\sigma(t) = \frac{1}{1 + e^{-t}}; \quad \quad \quad \frac{d}{dt} \sigma(t)= \sigma(t)(1-\sigma(t))$$

- *Tangente hiperbólica*:

$$\tanh(t) = \frac{e^t - e^{-t}}{e^t + e^{-t}}; \quad \quad \quad \frac{d}{dt} \tanh(t)= 1 - \tanh^2(t)$$

- *Entropía cruzada binaria*:

$$ \mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y_i}) - (1 - y_i) \log(1 - \hat{y_i}) \right] \quad \quad \quad \frac{\partial}{\partial\hat{y}} \mathcal{L} = \frac{1}{N}(\hat{y} - y)$$

## Funciones de activación y entropía cruzada

Primeramente debemos definir las funciones de activación y la función de pérdida que utilizaremos para este caso.

### <font color="teal">Actividad 1: Funciones</font>

<font color="teal">Comprueba el código de las funciones de funciones de activación sigmoide y tangente hiperbólica, así como de entropía cruzada binaria están correctamente definidas a partir de las ecuaciones.</font>

In [8]:
def sig(T: torch.Tensor):
    """Función de activación sigmoide"""
    return torch.reciprocal(1 + torch.exp(-1 * T))


def tanh(T: torch.Tensor):
    """Función de activación tangente hiperbólica."""
    E = torch.exp(T)
    e = torch.exp(-1 * T)
    return (E - e) * torch.reciprocal(E + e)


def bi_cross_ent_loss(y_pred, y, safe=True, epsilon=1e-7):
    """Función de pérdida de entropía cruzada binaria"""

    N = y.size()[0]  # tamaño del batch

    # Asegura que no haya valores indefinidos.
    if safe:
        y_pred = y_pred.clamp(epsilon, 1 - epsilon)

    B = (1-y) * torch.log(1 - y_pred) + y * torch.log(y_pred)
    return -1/N * torch.sum(B)

## Definición de la arquitectura de la red neuronal


Para resolver esta tarea, hemos definido una red neuronal consistente en dos capas ocultas, de 8 y 4 neuronas respectivamente. La capa de salida tiene una única neurona ya que la clasificación es binaria, es decir, puede ser solamente 0 ó 1.

Un grafo de computación de la neurona luce como lo siguiente:

![Red](https://github.com/jorgeanais/mlt2202/blob/main/ea1/s3/assets/redNN.png?raw=true)

### <font color="teal">Actividad 2: Construir una red usando los conceptos fundametales</font>

<font color="teal">Comprueba el código a continuación que sigue la arquitectura mostrada en el grafo anterior.</font>

Lo haremos desde primeros principios, como revisamos en el curso. Revisa los siguientes pasos:
1. Definimos nuestra red neuronal mediante la creación de una subclase que hereda de `nn.Module`.
2. Dentro del método `__init__` inicializamos todos los parámetros de las capas de la red neuronal (pesos y sesgos). Notar que inicializaremos los pesos de manera aleatorio, mientras que los sesgos en cero.
3. Definimos las operaciones de propagación en el método `forward`.
4. Finalmente definimos el paso de retropropagación en el método `backward`. Aquí calculamos los gradientes en sentido inverso usando la regla de la cadena. Guardamos los valores del gradiente para cada parámetro en su atributo `p.grad`.

**Nota**: El paso de retropropagación (`backward`) lo hemos incluido solo por propósitos pedagógicos, en la práctica no es necesario, ya que PyTorch lo calcula automáticamente a partir del grafo de computación que se define en `forward`.


In [9]:
class FFNN(torch.nn.Module):
    def __init__(self, d0=784, d1=64, d2=16):
        """
        Crea la red FFNN con 2 capas ocultas y una capa de salida.
        d0: dimensión de la capa de entrada
        d1: número de neuronas de la primera capa oculta
        d2: número de neuronas de la segunda capa oculta
        """
        super(FFNN, self).__init__()

        # Crea los tensores como parámetros
        self.W1 = torch.nn.Parameter(torch.randn(d0, d1))
        self.b1 = torch.nn.Parameter(torch.zeros(d1))
        self.W2 = torch.nn.Parameter(torch.randn(d1,d2))
        self.b2 = torch.nn.Parameter(torch.zeros(d2))
        self.U  = torch.nn.Parameter(torch.randn(d2,1))
        self.c  = torch.nn.Parameter(torch.zeros(1))


    def forward(self, x: torch.Tensor):
        # Calcula la pasada hacia adelante
        u1 = x @ self.W1 + self.b1
        h1 = tanh(u1)
        u2 = h1 @ self.W2 + self.b2
        h2 = sig(u2)
        u3 = h2 @ self.U + self.c
        y_pred = sig(u3)

        self._cache = [u1, u2]  # Guaradamos temporalmente los valores de u1 y u2

        return y_pred

    # Backpropagation
    def backward(self, x: torch.Tensor, y: torch.Tensor, y_pred: torch.Tensor):

        u1, u2 = self._cache  # recuperamos los valores de u1 y u2

        # tamaño del batch
        b = x.size()[0]

        # Estas son derivadas calculadas a mano
        dL_du3 = (1/b) * (y_pred - y)
        dL_dU  = sig(u2).t() @ dL_du3
        dL_dc  = torch.sum(dL_du3, 0)
        dL_dh2 = dL_du3 @ self.U.t()
        dL_du2 = dL_dh2 * sig(u2) * (1 - sig(u2))
        dL_dW2 = tanh(u1).t() @ dL_du2
        dL_db2 = torch.sum(dL_du2, 0)
        dL_dh1 = dL_du2 @ self.W2.t()
        dL_du1 = dL_dh1 * (1 - tanh(u1) * tanh(u1))
        dL_dW1 = x.t() @ dL_du1
        dL_db1 = torch.sum(dL_du1, 0)

        # Registra los valores de gradientes en cada tensor (que nos interesa)
        grads = [dL_dW1, dL_db1, dL_dW2, dL_db2, dL_dU, dL_dc]
        params = [self.W1, self.b1, self.W2, self.b2, self.U, self.c]
        for p, g in zip(params, grads):
            p.grad = g

    def num_parameters(self):
        total = 0
        for p in self.parameters():
            total += p.numel()
        return total

## Preparación de la ingesta de datos

Ahora prepararemos el código que se encuarga del procesamiento de los datos. Para ello haremos uso de las primitivas: `torch.utils.data.Dataset` y `torch.utils.data.DataLoader`. Esto con el fin de que el código de nuestro conjunto de datos esté desacoplado del código de entrenamiento de nuestro modelo para una mejor legibilidad y modularidad.

### <font color="teal">Actividad 3. Generar el Dataset</font>

<font color="teal">Revisa como se ha construido la siguiente clase.</font>


1.   Primero definimos el método `__init__` donde se lee el archivo CSV y se transforman los datos y sus respectivas etiquetas a "tensores" de PyTorch.
2.   El método `__getitem__` permite posteriormente extraer los ejemplos del dataset para el entrenamiento.
3.   Finalmente, `__len__` simplemente retorna el número de ejemplos.



In [10]:
class CustomDataSet(Dataset):
    def __init__(self, csv_path: Path):
        """Lee el archivo CSV con los datos y genera un Dataset"""
        df = pd.read_csv(csv_path)

        labels = torch.tensor(df["label"].values, dtype=torch.long)
        self.labels = labels.unsqueeze(1)  # shape: (N, 1)


        pixel_cols = [c for c in df.columns if c.startswith("pixel_")]
        pixels = df[pixel_cols].values.astype(np.float32) / 255.0  # Normalización
        self.flatten_images = torch.tensor(pixels)  # shape: (N, 784)

        self.num_features = len(pixel_cols)

    # Debemos definir __len__ para retornar el tamaño del dataset
    def __len__(self):
        return len(self.labels)

    # Debemos definir __getitem__ para retornar el i-ésimo ejemplo en nuestro dataset.
    def __getitem__(self, idx):
        flatten_image = self.flatten_images[idx]  # shape: (784,)
        label = self.labels[idx]

        return flatten_image, label

## Proceso de entrenamiento

Ahora denemos crear el bucle de entrenamiento de la red, es decir, debemos realizar los pasos de forward y backward para nustros datos, y en cada pasada actualizar los valores de los pesos y sesgos utilizando un algoritmo de optimización como el descenso del gradiente según la pérdida que obtengamos.

### <font color="teal">Actividad 4: Bucle de entrenamiento</font>

 <font color="teal">Identifica en el código los principales pasos del entrenamiento.</font>

Las principales partes del bucle son las siguientes:


1.   Instanciar la red y cargar los datos
2.   Entrenar la red haciendo por cada lote (*batch*) de datos (ciclo `for`):  
  2.1 Calcular la propagación hacia adelante  
  2.2 Calcular la pérdida  
  2.3 Calcular la retropropagación  
  2.4 Actualizar los parámetros usando el descenso del gradiente.  
3. Repetir lo anterior N épocas.

Hemos incluido que nos reporte el valor acierto que tiene nuestra red y el tiempo que tarda en cada época.



In [11]:
def loop_FFNN(
    dataset: Dataset,
    batch_size: int,   # tamaño del lote
    d1: int,  # número de neuronas en la capa 1
    d2: int,  # número de neuronas en la capa 2
    lr: float,  # tasa de aprendizaje
    epochs: int,
    run_in_GPU: bool=True,
    reports_every: int=1,
):
    # Define un tipo para los tensores según si correrá en la GPU o no
    device = 'cuda' if run_in_GPU else 'cpu'

    # d0 es la cantidad de `features` del dataset (pixeles de cada imagen)
    d0 = dataset.num_features

    # Cantidad de ejemplos
    N = len(dataset)

    # Instanciamos la red
    red = FFNN(d0, d1, d2)

    # Cargar la red en la GPU o CPU según elección
    red.to(device)

    # Mostrar la cantidad de parámetros
    print(f"Cantidad de parámetros: {red.num_parameters()}")

    # Crea un dataloader desde el dataset
    data = DataLoader(dataset, batch_size, shuffle=True)

    # Comienza el entrenamiento
    tiempo_epochs = 0
    for e in range(1, epochs + 1):
        inicio_epoch = time.process_time()

        for (x, y) in data:
            # Asegura de pasarlos a la GPU si fuera necesario
            x, y = x.to(device), y.to(device)

            # Computa la pasada hacia adelante (forward)
            y_pred = red.forward(x)

            # Computa la función de pérdida
            L = bi_cross_ent_loss(y_pred, y)

            # Computa los gradientes hacia atrás (backpropagation)
            red.backward(x, y, y_pred)


            # Descenso de gradiente para actualizar los parámetros
            for p in red.parameters():
                p.data = p.data - lr * p.grad

        tiempo_epochs += time.process_time() - inicio_epoch

        # Reporta el acierto cada "reports_every" cantidad de épocas
        if e % reports_every == 0:

            # Calcula la certeza de las predicciones sobre todo el conjunto
            X = dataset.flatten_images.to(device)
            Y = dataset.labels.to(device)

            # Predice usando la red
            Y_PRED = red.forward(X)

            # Calcula la pérdida de todo el conjunto
            L_total = bi_cross_ent_loss(Y_PRED, Y)

            # Elige una clase dependiendo del valor de Y_PRED
            Y_PRED_BIN = (Y_PRED >= 0.5).float()

            correctos = torch.sum(Y_PRED_BIN == Y).item()
            acc = (correctos / N) * 100

            sys.stdout.write(
                f"Epoch:{e:03d} Acc:{acc:.2f} Loss:{L_total:.4f} Tiempo/epoch:{tiempo_epochs/e:.3f}s\n"
            )



## Entrenando la red

**¡Enhorabuena!**

Luego un arduo trabajo hemos logrado construir una red neuronal, un conjunto de datos y un proceso de entrenamiento que potencialmente puede aprender a clasificar los dígitos escritos a mano.

Ahora estamos listos para entrenar nuestra red y que aprenda de nuestros datos. Para lograrlo tenemos que:
1. Cargar el archivo CSV que contiene los datos usando la clase `CustomDataSet` que hemos definido anteriormente.
2. Ejecutar la función de entrenamiento por 10 épocas.

Nota: Si estas ejecutando este notebook eng Google Colab, asegúrate de que el *runtime type* tenga activado la opción de  *Hardware accelerator* de tipo GPU, tal como `T4 GPU`.

In [12]:
csv_path = Path("/content/mnist_digits_0_1.csv")
dataset = CustomDataSet(csv_path)

In [13]:
loop_FFNN(
    dataset=dataset,
    batch_size=8,   # tamaño del lote
    d1=8,            # número de neuronas en la capa 1
    d2=4,            # número de neuronas en la capa 2
    lr=0.001,        # tasa de aprendizaje
    epochs=10,       # numero de epocas
    run_in_GPU=True,
    reports_every=1,
  )

Cantidad de parámetros: 6321
Epoch:001 Acc:57.31 Loss:0.7928 Tiempo/epoch:4.306s
Epoch:002 Acc:76.58 Loss:0.5813 Tiempo/epoch:4.040s
Epoch:003 Acc:84.85 Loss:0.4619 Tiempo/epoch:3.902s
Epoch:004 Acc:89.05 Loss:0.3796 Tiempo/epoch:3.951s
Epoch:005 Acc:91.93 Loss:0.3214 Tiempo/epoch:4.052s
Epoch:006 Acc:94.28 Loss:0.2784 Tiempo/epoch:3.974s
Epoch:007 Acc:95.54 Loss:0.2448 Tiempo/epoch:4.026s
Epoch:008 Acc:96.33 Loss:0.2184 Tiempo/epoch:4.010s
Epoch:009 Acc:96.91 Loss:0.1970 Tiempo/epoch:4.015s
Epoch:010 Acc:97.33 Loss:0.1792 Tiempo/epoch:4.070s


### <font color="teal"> Actividad 5</font>

<font color="teal">Utilizando el código anterior, responde las siguientes preguntas:</font>

1. ¿Cuántos parámetros tiene la red?
2. ¿Qué pasa al aumentar el tamaño del lote (`batch_size`) de 8 a 16 dejando el número de épocas fijas a 10? Haga el experimento.
3. En el caso anterior ¿Disminuye la cantidad de veces que se realiza el descenso del gradiente para actualizar los parámetros?
4. ¿Cómo podría mejorar el Accuracy obtenido? Experimente y saque sus conclusiones.
5. ¿Qué sucede al variar el hiperparámetro de tasa de aprendizaje o *learning rate* `lr`? Experimente.
6. Ejecute los siguientes experimentos, utilizando los parámetros indicados en la tabla y complete los valores de tiempo de ejecución por época, el número de parámetros y el accuracy según sus resultados.

|              | Exp1   | Exp2   | Exp3    | Exp4    | Exp5    | Exp6    |
|--------------|--------|--------|---------|---------|---------|---------|
| device       | GPU    | CPU    | GPU     | CPU     | GPU     | CPU     |
| epochs       | 10     | 10     | 10      | 10      | 10      | 10      |
| batch size   | 8      | 8      | 16      | 16      | 8       | 8       |
| d1           | 8      | 8      | 8       | 8       | 512     | 512     |
| d2           | 4      | 4      | 4       | 4       | 64      |   64    |
| tiempo/epoca |        |        |         |         |         |         |
| N params     |        |        |         |         |         |         |
| acc          |        |        |         |         |         |         |

7. Con respecto a los experimentos 1, 2, 3 y 4 ¿Qué conclusión puede sacar en cuanto al tiempo que demora por época?  
8. Con respecto a los experimentos 6 y 7, ¿cuál demora menos?  
9. ¿Siempre es más rápido GPU? ¿por qué pasa esto?
10. ¿Cuál red obtuvo mejor accuracy?

In [16]:
loop_FFNN(
    dataset=dataset,
    batch_size=16,   # tamaño del lote
    d1=8,            # número de neuronas en la capa 1
    d2=4,            # número de neuronas en la capa 2
    lr=0.001,        # tasa de aprendizaje
    epochs=10,       # numero de epocas
    run_in_GPU=True,
    reports_every=1,
  )

Cantidad de parámetros: 6321
Epoch:001 Acc:66.14 Loss:0.6582 Tiempo/epoch:1.950s
Epoch:002 Acc:72.29 Loss:0.6160 Tiempo/epoch:2.237s
Epoch:003 Acc:77.06 Loss:0.5771 Tiempo/epoch:2.169s
Epoch:004 Acc:81.14 Loss:0.5409 Tiempo/epoch:2.142s
Epoch:005 Acc:84.03 Loss:0.5072 Tiempo/epoch:2.101s
Epoch:006 Acc:86.87 Loss:0.4755 Tiempo/epoch:2.075s
Epoch:007 Acc:89.42 Loss:0.4458 Tiempo/epoch:2.056s
Epoch:008 Acc:90.74 Loss:0.4183 Tiempo/epoch:2.115s
Epoch:009 Acc:91.96 Loss:0.3927 Tiempo/epoch:2.116s
Epoch:010 Acc:93.39 Loss:0.3691 Tiempo/epoch:2.098s


In [17]:
loop_FFNN(
    dataset=dataset,
    batch_size=16,   # tamaño del lote
    d1=8,            # número de neuronas en la capa 1
    d2=4,            # número de neuronas en la capa 2
    lr=0.001,        # tasa de aprendizaje
    epochs=10,       # numero de epocas
    run_in_GPU=False,
    reports_every=1,
  )

Cantidad de parámetros: 6321
Epoch:001 Acc:83.55 Loss:0.5420 Tiempo/epoch:1.424s
Epoch:002 Acc:87.71 Loss:0.5098 Tiempo/epoch:1.389s
Epoch:003 Acc:90.69 Loss:0.4852 Tiempo/epoch:1.401s
Epoch:004 Acc:91.88 Loss:0.4633 Tiempo/epoch:1.445s
Epoch:005 Acc:92.83 Loss:0.4428 Tiempo/epoch:1.531s
Epoch:006 Acc:93.53 Loss:0.4231 Tiempo/epoch:1.534s
Epoch:007 Acc:94.03 Loss:0.4043 Tiempo/epoch:1.530s
Epoch:008 Acc:94.61 Loss:0.3864 Tiempo/epoch:1.554s
Epoch:009 Acc:95.08 Loss:0.3693 Tiempo/epoch:1.549s
Epoch:010 Acc:95.55 Loss:0.3531 Tiempo/epoch:1.545s


In [18]:
loop_FFNN(
    dataset=dataset,
    batch_size=8,   # tamaño del lote
    d1=512,            # número de neuronas en la capa 1
    d2=64,            # número de neuronas en la capa 2
    lr=0.001,        # tasa de aprendizaje
    epochs=10,       # numero de epocas
    run_in_GPU=True,
    reports_every=1,
  )

Cantidad de parámetros: 434817
Epoch:001 Acc:82.96 Loss:0.3868 Tiempo/epoch:4.635s
Epoch:002 Acc:91.61 Loss:0.2116 Tiempo/epoch:4.309s
Epoch:003 Acc:94.74 Loss:0.1461 Tiempo/epoch:4.199s
Epoch:004 Acc:95.98 Loss:0.1127 Tiempo/epoch:4.204s
Epoch:005 Acc:96.74 Loss:0.0926 Tiempo/epoch:4.078s
Epoch:006 Acc:97.31 Loss:0.0791 Tiempo/epoch:3.993s
Epoch:007 Acc:97.71 Loss:0.0693 Tiempo/epoch:4.021s
Epoch:008 Acc:97.95 Loss:0.0619 Tiempo/epoch:3.968s
Epoch:009 Acc:98.25 Loss:0.0560 Tiempo/epoch:3.928s
Epoch:010 Acc:98.42 Loss:0.0512 Tiempo/epoch:3.954s


In [19]:
loop_FFNN(
    dataset=dataset,
    batch_size=8,   # tamaño del lote
    d1=512,            # número de neuronas en la capa 1
    d2=64,            # número de neuronas en la capa 2
    lr=0.001,        # tasa de aprendizaje
    epochs=10,       # numero de epocas
    run_in_GPU=False,
    reports_every=1,
  )

Cantidad de parámetros: 434817
Epoch:001 Acc:89.21 Loss:0.2712 Tiempo/epoch:4.137s
Epoch:002 Acc:93.78 Loss:0.1597 Tiempo/epoch:4.123s
Epoch:003 Acc:95.73 Loss:0.1157 Tiempo/epoch:4.364s
Epoch:004 Acc:96.68 Loss:0.0922 Tiempo/epoch:4.308s
Epoch:005 Acc:97.29 Loss:0.0774 Tiempo/epoch:4.265s
Epoch:006 Acc:97.69 Loss:0.0671 Tiempo/epoch:4.322s
Epoch:007 Acc:97.96 Loss:0.0595 Tiempo/epoch:4.311s
Epoch:008 Acc:98.18 Loss:0.0536 Tiempo/epoch:4.341s
Epoch:009 Acc:98.38 Loss:0.0489 Tiempo/epoch:4.335s
Epoch:010 Acc:98.54 Loss:0.0450 Tiempo/epoch:4.328s


In [20]:
loop_FFNN(
    dataset=dataset,
    batch_size=8,   # tamaño del lote
    d1=1024,            # número de neuronas en la capa 1
    d2=512,            # número de neuronas en la capa 2
    lr=0.001,        # tasa de aprendizaje
    epochs=10,       # numero de epocas
    run_in_GPU=True,
    reports_every=1,
  )

Cantidad de parámetros: 1329153
Epoch:001 Acc:92.56 Loss:0.3564 Tiempo/epoch:4.255s
Epoch:002 Acc:96.45 Loss:0.1667 Tiempo/epoch:4.045s
Epoch:003 Acc:97.54 Loss:0.1077 Tiempo/epoch:3.915s
Epoch:004 Acc:98.21 Loss:0.0780 Tiempo/epoch:3.991s
Epoch:005 Acc:98.56 Loss:0.0600 Tiempo/epoch:3.924s
Epoch:006 Acc:98.84 Loss:0.0478 Tiempo/epoch:3.880s
Epoch:007 Acc:99.05 Loss:0.0390 Tiempo/epoch:3.909s
Epoch:008 Acc:99.17 Loss:0.0325 Tiempo/epoch:3.900s
Epoch:009 Acc:99.30 Loss:0.0274 Tiempo/epoch:3.877s
Epoch:010 Acc:99.38 Loss:0.0235 Tiempo/epoch:3.891s


In [21]:
loop_FFNN(
    dataset=dataset,
    batch_size=8,   # tamaño del lote
    d1=1024,            # número de neuronas en la capa 1
    d2=512,            # número de neuronas en la capa 2
    lr=0.001,        # tasa de aprendizaje
    epochs=10,       # numero de epocas
    run_in_GPU=False,
    reports_every=1,
  )

Cantidad de parámetros: 1329153
Epoch:001 Acc:94.36 Loss:0.2416 Tiempo/epoch:13.580s
Epoch:002 Acc:96.95 Loss:0.1332 Tiempo/epoch:14.364s
Epoch:003 Acc:97.90 Loss:0.0930 Tiempo/epoch:14.849s
Epoch:004 Acc:98.38 Loss:0.0714 Tiempo/epoch:15.215s
Epoch:005 Acc:98.71 Loss:0.0577 Tiempo/epoch:15.345s
Epoch:006 Acc:98.86 Loss:0.0480 Tiempo/epoch:15.420s
Epoch:007 Acc:99.02 Loss:0.0406 Tiempo/epoch:15.552s
Epoch:008 Acc:99.21 Loss:0.0347 Tiempo/epoch:15.596s
Epoch:009 Acc:99.30 Loss:0.0300 Tiempo/epoch:15.634s
Epoch:010 Acc:99.40 Loss:0.0262 Tiempo/epoch:15.736s


|              | Exp1   | Exp2   | Exp3    | Exp4    | Exp5    | Exp6    |
|--------------|--------|--------|---------|---------|---------|---------|
| device       | GPU    | CPU    | GPU     | CPU     | GPU     | CPU     |
| epochs       | 10     | 10     | 10      | 10      | 10      | 10      |
| batch size   | 8      | 8      | 16      | 16      | 8       | 8       |
| d1           | 8      | 8      | 8       | 8       | 16      | 16      |
| d2           | 4      | 4      | 4       | 4       | 8       | 8       |
| tiempo/epoca |  3.7   | 2.48   | 3.95    | 4.3     | 3.89    | 15.73   |
| N params     |  6321  | 6321   | 434817  | 434817  | 1329153 | 1329153 |
| acc          |  95.9  | 96.2   | 98.42   | 98.54   | 99.38   | 99.40   |
